# Challenge 02 - Cats vs Dogs (HOG features)

Binary classification of RGB images of cats (`0`) and dogs (`1`) using HOG features only: no external images and no data augmentation. The predictions are saved in `Es4Predictions.txt`.

## Setup

In [1]:
import os
os.environ['LOKY_MAX_CPU_COUNT'] = '4'

import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from joblib import Memory
from skimage import color, feature, transform
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import VotingClassifier
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings('ignore', message='Persisting input arguments took')

# Configuration
db_path = '../../DBs/CaniGatti_ML18'
train_file = 'BinaryTrainingSet.txt'
test_file = 'Unlabeled_BinaryTestSet.txt'
exp_path = '../Experiments'
result_path = '../Es4Predictions.txt'

# HOG and pipeline parameters
image_sides = [176, 208]
orientations = 12
cell_side = 16
block_side = 3
alpha = 0.5

# Disk cache: re-running a function with the same parameters is instantaneous
memory = Memory(exp_path, verbose=0)

## Helper Functions

Image loading, shuffling and resizing (from the course `ml_utilities` module, cached on disk with joblib), HOG extraction and the export of the predictions.

In [2]:
def shuffle_in_unison(elements, seed):
    """Shuffle every array in `elements` with the same permutation."""
    prev_state = np.random.get_state()
    np.random.seed(seed)
    rng_state = np.random.get_state()
    for x in elements:
        np.random.set_state(rng_state)
        np.random.shuffle(x)
    np.random.set_state(prev_state)


def internal_load_labeled_dataset(filelist_name, db_folder_path):
    db_folder_path = Path(db_folder_path)
    files, labels = [], []
    with open(db_folder_path / filelist_name) as f:
        for line in f:
            path, label = line.split()
            files.append(db_folder_path / path)
            labels.append(int(label))
    return [plt.imread(file_path) for file_path in files], np.array(labels)


def internal_load_unlabeled_dataset(filelist_name, db_folder_path):
    db_folder_path = Path(db_folder_path)
    with open(db_folder_path / filelist_name) as f:
        files = [db_folder_path / line.strip() for line in f]
    return [plt.imread(file_path) for file_path in files]


def internal_resize_images(imgs, size_h, size_w):
    """Resize RGB images (h, w, 3) to (size_h, size_w, 3)."""
    resized = np.empty((len(imgs), size_h, size_w, 3), dtype=np.float32)
    for i, img in enumerate(imgs):
        resized[i] = transform.resize(img, (size_h, size_w), mode='reflect', anti_aliasing=True)
    return resized


def cached(function, cache):
    return cache.cache(function) if cache is not None else function


def load_labeled_dataset(filelist_name, db_folder_path, cache=None):
    return cached(internal_load_labeled_dataset, cache)(filelist_name, db_folder_path)


def load_unlabeled_dataset(filelist_name, db_folder_path, cache=None):
    return cached(internal_load_unlabeled_dataset, cache)(filelist_name, db_folder_path)


def resize_images(imgs, size_h, size_w, cache=None):
    return cached(internal_resize_images, cache)(imgs, size_h, size_w)


def extract_hog_features(images, side):
    """Resize to side x side, convert to grayscale, and extract HOG features."""
    resized = resize_images(images, side, side, cache=memory)
    hog_features = []
    for img in resized:
        hog = feature.hog(color.rgb2gray(img),
                          orientations=orientations,
                          pixels_per_cell=(cell_side, cell_side),
                          cells_per_block=(block_side, block_side),
                          block_norm='L2-Hys')
        hog_features.append(hog)
    return np.array(hog_features)


def power_transform(X, alpha=0.5):
    """Element-wise x**alpha (HOG L2-Hys is >= 0; clipping guards against -0.0)."""
    return np.power(np.clip(X, 0, None), alpha)


def save_predictions(predictions, path):
    """One integer label per line, no header."""
    with open(path, 'w') as f:
        for prediction in predictions:
            f.write(f'{int(prediction)}\n')
    print(f'Predictions successfully saved to {path}')

## Data Loading

The dataset contains 900 labeled RGB images plus an unlabeled test set. The training images are shuffled with a fixed seed and kept in their original size, so that they can be resized at different resolutions. HOG features are then extracted at 176×176 and 208×208 px (12 orientations, 16×16 cells, 3×3 cells per block), with the same preprocessing for training and test images.

In [3]:
start = time.time()
train_orig_x, train_y = load_labeled_dataset(train_file, db_path, cache=memory)
test_orig_x = load_unlabeled_dataset(test_file, db_path, cache=memory)
shuffle_in_unison([train_orig_x, train_y], seed=1234)
print('Loaded %d training and %d test images in %.2f s.' % (len(train_orig_x), len(test_orig_x), time.time() - start))
print('Cats: %d, dogs: %d' % (np.count_nonzero(train_y == 0), np.count_nonzero(train_y == 1)))

# HOG at two resolutions, identical for training and test
train_parts = [extract_hog_features(train_orig_x, side) for side in image_sides]
train_x = np.hstack(train_parts)
test_x = np.hstack([extract_hog_features(test_orig_x, side) for side in image_sides])
print('HOG train/test shape:', train_x.shape, test_x.shape)

Loaded 900 training and 150 test images in 2.96 s.
Cats: 509, dogs: 391
HOG train/test shape: (900, 21816) (150, 21816)
